In [1]:
import json
import random
import re
import traceback
from pathlib import Path

import matlab.engine


In [ ]:
# =========================================
# 1. 사용자 설정
# =========================================
MODEL_NAME = "Matlab/P1"
N_SAMPLES = 5
SAVE_PATH = Path("msd_sampled_results.json")

# 샘플링 범위
PARAM_SPACE = {
    "Kp": (5.0, 20.0),
    "Ki": (1.0, 15.0),
    "Kd": (0.1, 5.0),
}

# 고정 파라미터
FIXED_PARAMS = {
    "m": 1.0,
    "c": 2.0,
    "k": 5.0,
}

# raw signal 저장 여부
SAVE_SIGNALS = True

In [3]:
# =========================================
# 2. 변환 유틸
# =========================================
def matlab_scalar_to_float(x):
    """
    MATLAB scalar / matlab.double / string representation를 Python float로 변환
    """
    if isinstance(x, (int, float)):
        return float(x)

    if isinstance(x, str):
        s = x.strip()
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
        if match:
            return float(match.group(0))
        raise ValueError(f"Could not parse numeric value from: {x!r}")

    try:
        data = list(x)
        if len(data) == 1:
            return matlab_scalar_to_float(data[0])
    except Exception:
        pass

    return matlab_scalar_to_float(str(x))


def matlab_vector_to_1d_list(x, ndigits=8):
    """
    matlab.double 벡터 / Python list / scalar를 JSON 저장용 list로 변환
    """
    if isinstance(x, (int, float, str)):
        return [round(matlab_scalar_to_float(x), ndigits)]

    data = list(x)

    if len(data) == 0:
        return []

    if isinstance(data[0], list) and len(data[0]) == 1:
        return [round(matlab_scalar_to_float(row[0]), ndigits) for row in data]

    if isinstance(data[0], list) and len(data) == 1:
        return [round(matlab_scalar_to_float(v), ndigits) for v in data[0]]

    if not isinstance(data[0], list):
        return [round(matlab_scalar_to_float(v), ndigits) for v in data]

    return [[round(matlab_scalar_to_float(v), ndigits) for v in row] for row in data]


In [4]:
# =========================================
# 3. 시뮬레이션 유틸
# =========================================
def sample_params(param_space: dict, fixed_params: dict) -> dict:
    sampled = {
        key: random.uniform(low, high)
        for key, (low, high) in param_space.items()
    }
    sampled.update(fixed_params)
    return sampled


def set_workspace_params(eng, params: dict):
    for key, value in params.items():
        eng.workspace[key] = float(value)


def run_simulation(eng, model_name: str):
    eng.eval(f"out = sim('{model_name}');", nargout=0)


def compute_stepinfo_position(eng):
    eng.eval("info_pos = stepinfo(out.pos_out(:), out.t_out(:));", nargout=0)

    return {
        "RiseTime": matlab_scalar_to_float(eng.eval("info_pos.RiseTime", nargout=1)),
        "TransientTime": matlab_scalar_to_float(eng.eval("info_pos.TransientTime", nargout=1)),
        "SettlingTime": matlab_scalar_to_float(eng.eval("info_pos.SettlingTime", nargout=1)),
        "SettlingMin": matlab_scalar_to_float(eng.eval("info_pos.SettlingMin", nargout=1)),
        "SettlingMax": matlab_scalar_to_float(eng.eval("info_pos.SettlingMax", nargout=1)),
        "Overshoot": matlab_scalar_to_float(eng.eval("info_pos.Overshoot", nargout=1)),
        "Undershoot": matlab_scalar_to_float(eng.eval("info_pos.Undershoot", nargout=1)),
        "Peak": matlab_scalar_to_float(eng.eval("info_pos.Peak", nargout=1)),
        "PeakTime": matlab_scalar_to_float(eng.eval("info_pos.PeakTime", nargout=1)),
    }


def collect_signals(eng):
    return {
        "t_out": matlab_vector_to_1d_list(eng.eval("out.t_out", nargout=1)),
        "pos_out": matlab_vector_to_1d_list(eng.eval("out.pos_out", nargout=1)),
        "vel_out": matlab_vector_to_1d_list(eng.eval("out.vel_out", nargout=1)),
    }


def run_one_case(eng, model_name: str, params: dict, save_signals: bool = True) -> dict:
    set_workspace_params(eng, params)
    run_simulation(eng, model_name)

    result = {
        "params": {k: float(v) for k, v in params.items()},
        "position_metrics": compute_stepinfo_position(eng),
    }

    if save_signals:
        result["signals"] = collect_signals(eng)

    return result

In [5]:
# =========================================
# 4. 메인 실행
# =========================================
def main():
    eng = matlab.engine.start_matlab()

    try:
        eng.load_system(MODEL_NAME, nargout=0)

        stepinfo_path = eng.eval("which('stepinfo','-all')", nargout=1)
        has_control_toolbox = eng.eval("license('test','Control_Toolbox')", nargout=1)

        print("stepinfo path:")
        print(stepinfo_path)
        print("Control Toolbox available:", has_control_toolbox)

        if not has_control_toolbox:
            raise RuntimeError("Control System Toolbox 라이선스를 사용할 수 없습니다.")

        results = []

        for i in range(N_SAMPLES):
            params = sample_params(PARAM_SPACE, FIXED_PARAMS)

            try:
                result = run_one_case(
                    eng=eng,
                    model_name=MODEL_NAME,
                    params=params,
                    save_signals=SAVE_SIGNALS,
                )
                results.append(result)

                print(f"[{i+1}/{N_SAMPLES}] success")
                print("  params:", result["params"])
                print("  metrics:", result["position_metrics"])

            except Exception as e:
                print(f"[{i+1}/{N_SAMPLES}] failed: {e}")
                traceback.print_exc()

        with open(SAVE_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        print(f"\nSaved JSON: {SAVE_PATH.resolve()}")
        print(f"Total successful cases: {len(results)}")

    finally:
        try:
            eng.quit()
        except Exception:
            pass


if __name__ == "__main__":
    main()


stepinfo path:
['C:\\Program Files\\MATLAB\\R2024a\\toolbox\\shared\\controllib\\engine\\stepinfo.m', 'C:\\Program Files\\MATLAB\\R2024a\\toolbox\\control\\ctrlanalysis\\@DynamicSystem\\stepinfo.m']
Control Toolbox available: 1.0
[1/5] success
  params: {'Kp': 19.47432618308725, 'Ki': 14.77356384715383, 'Kd': 2.0241899393308045, 'm': 1.0, 'c': 2.0, 'k': 5.0}
  metrics: {'RiseTime': 0.2959803403050414, 'TransientTime': 3.332474433624184, 'SettlingTime': 3.332474433624184, 'SettlingMin': 0.8302559335880116, 'SettlingMax': 1.146270481500137, 'Overshoot': 14.626785690787457, 'Undershoot': -0.0, 'Peak': 1.146270481500137, 'PeakTime': 0.6152766367440066}
[2/5] success
  params: {'Kp': 15.104720411558668, 'Ki': 5.9471346532907345, 'Kd': 3.659749227600582, 'm': 1.0, 'c': 2.0, 'k': 5.0}
  metrics: {'RiseTime': 0.3940726465081055, 'TransientTime': 7.994511976879592, 'SettlingTime': 7.994511976879592, 'SettlingMin': 0.8182351042589445, 'SettlingMax': 1.000019364047435, 'Overshoot': 0.002215586248